# Model 2: 100 training observations



In [1]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
wine = pd.read_parquet("df_with_color.parquet")
print("Loaded reviews:", len(wine))

Loaded reviews: 169407


## Select the subset and create separate samples



In [2]:
expected_colors = {"Cabernet Sauvignon": "red", "Chardonnay": "white"}
subset = wine.loc[
    wine["province"].eq("California")
    & wine["variety"].isin(expected_colors)
].dropna(subset=["description", "color"]).copy()
subset["description_key"] = (
    subset["description"].str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()
)
subset = subset.loc[subset["description_key"].ne("")].copy()
assert subset["color"].eq(subset["variety"].map(expected_colors)).all(), "Check color mapping."
conflicting = subset.groupby("description_key")["color"].nunique()
subset = subset.loc[~subset["description_key"].isin(conflicting[conflicting > 1].index)]
subset = subset.drop_duplicates("description_key").reset_index(drop=True)

training_parts, validation_parts = [], []
for color in ["red", "white"]:
    pool = subset.loc[subset["color"].eq(color)]
    assert len(pool) >= 55, f"Not enough {color} observations."
    selected = pool.sample(n=55, random_state=SEED)
    training_parts.append(selected.iloc[:5])
    validation_parts.append(selected.iloc[5:])

train = pd.concat(training_parts).sample(frac=1, random_state=SEED).copy()
validation = pd.concat(validation_parts).sample(frac=1, random_state=SEED).copy()
assert len(train) == 10
assert set(train["description_key"]).isdisjoint(validation["description_key"])
print("Training counts:\n", train["color"].value_counts())
print("\nValidation counts:\n", validation["color"].value_counts())

Training counts:
 color
white    5
red      5
Name: count, dtype: int64

Validation counts:
 color
white    50
red      50
Name: count, dtype: int64


## Recreate Model 1 for comparison

Fit the original ten-review model with its unchanged settings. This reproduces the reference inside this notebook without depending on another notebook's kernel state.

In [3]:
X_train, y_train = train["description"], train["color"]
X_val, y_val = validation["description"], validation["color"]

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

model = make_pipeline(
    TfidfVectorizer(max_features=100, ngram_range=(1, 1), min_df=1),
    LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
)
model.fit(X_train, y_train)
print("Reviews used to fit the model:", len(X_train))
print("TF-IDF vocabulary size:", len(model[0].get_feature_names_out()))

Reviews used to fit the model: 10
TF-IDF vocabulary size: 100


## Model 2: Increase training data from 10 to 100 reviews

Model 1's 100% training accuracy versus 78% validation accuracy suggests overfitting. Add 90 training reviews (45 per color) from the unused pool, retaining the original ten. Keep the same California Cabernet Sauvignon/Chardonnay scope, the same 100 validation reviews, and the same model settings. TF-IDF is refitted on the enlarged training set, so its learned vocabulary and weights can change even though its settings stay fixed.

This experiment tests whether more training data improves generalization. The validation reviews remain excluded from training; they are not a final test set.

In [4]:
# Exclude both the original training reviews and the fixed validation reviews.
used_keys = set(train["description_key"]) | set(validation["description_key"])
unused = subset.loc[~subset["description_key"].isin(used_keys)]
additional_parts = []
for color in ["red", "white"]:
    pool = unused.loc[unused["color"].eq(color)]
    assert len(pool) >= 45, f"Not enough unused {color} reviews."
    additional_parts.append(pool.sample(n=45, random_state=SEED))

train_100 = pd.concat([train, *additional_parts]).sample(frac=1, random_state=SEED).copy()
assert len(train_100) == 100
assert train_100["description_key"].is_unique
assert set(train["description_key"]).issubset(set(train_100["description_key"]))
assert set(train_100["description_key"]).isdisjoint(validation["description_key"])
print("Model 2 training counts:\n", train_100["color"].value_counts())
print("Unchanged validation counts:\n", validation["color"].value_counts())

Model 2 training counts:
 color
white    50
red      50
Name: count, dtype: int64
Unchanged validation counts:
 color
white    50
red      50
Name: count, dtype: int64


In [5]:
# Clone preserves Model 1's settings without reusing its fitted state.
from sklearn.base import clone

model_2 = clone(model)
model_2.fit(train_100["description"], train_100["color"])

comparison_rows = []
for name, estimator, training_data in [
    ("Model 1: 10 reviews", model, train),
    ("Model 2: 100 reviews", model_2, train_100),
]:
    for split, frame in [("training", training_data), ("validation", validation)]:
        predicted = estimator.predict(frame["description"])
        comparison_rows.append({
            "model": name, "split": split, "n": len(frame),
            "accuracy": accuracy_score(frame["color"], predicted),
            "macro_f1": f1_score(frame["color"], predicted, average="macro", zero_division=0),
        })
comparison = pd.DataFrame(comparison_rows)
print(comparison.to_string(index=False))

predictions_2 = model_2.predict(validation["description"])
print("\nModel 2 validation classification report:")
print(classification_report(validation["color"], predictions_2, digits=3, zero_division=0))
print("Confusion matrix: rows = actual, columns = predicted")
print(pd.DataFrame(
    confusion_matrix(validation["color"], predictions_2, labels=["red", "white"]),
    index=["actual red", "actual white"],
    columns=["predicted red", "predicted white"],
))

scores = comparison.pivot(index="model", columns="split", values="accuracy")
print("\nTraining minus validation accuracy (percentage points):")
print(((scores["training"] - scores["validation"]) * 100).round(1).to_string())
change = scores.loc["Model 2: 100 reviews", "validation"] - scores.loc["Model 1: 10 reviews", "validation"]
print(f"\nValidation accuracy change: {change * 100:+.1f} percentage points")

               model      split   n  accuracy  macro_f1
 Model 1: 10 reviews   training  10      1.00  1.000000
 Model 1: 10 reviews validation 100      0.78  0.776786
Model 2: 100 reviews   training 100      0.98  0.980000
Model 2: 100 reviews validation 100      0.93  0.929993

Model 2 validation classification report:
              precision    recall  f1-score   support

         red      0.939     0.920     0.929        50
       white      0.922     0.940     0.931        50

    accuracy                          0.930       100
   macro avg      0.930     0.930     0.930       100
weighted avg      0.930     0.930     0.930       100

Confusion matrix: rows = actual, columns = predicted
              predicted red  predicted white
actual red               46                4
actual white              3               47

Training minus validation accuracy (percentage points):
model
Model 1: 10 reviews     22.0
Model 2: 100 reviews     5.0

Validation accuracy change: +15.0 percen

## Record Model 2's outcome

We see a big jump in validation. Our training accuracy has slightly decreased, from 100 to 98%, and our validation has gone from 78% to 93%. Seeing that there still is a gap of 5%, next we're going to increase to 1000 samples, to see if this continues to improve performance, or levels off. 